# Additional Instructions

Trying to improve LLMGraphTransformer peformance with examples.

In [1]:
import os

import networkx as nx
from langchain.chains import GraphQAChain
from langchain_core.documents import Document
from langchain_community.graphs.networkx_graph import NetworkxEntityGraph
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from lib.llm import LLMGraphTransformer
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
import json
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
import pandas as pd
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
import chroma
import json_repair

In [2]:

llm_model = "llama3.2:latest"
llm = ChatOllama(
   model=llm_model,
   temperature=0,
   # other params...
)


with open("./data/training_modeling_papers.json", "r") as f:
    train_data = json.load(f)

with open("./data/modeling_papers.json", "r") as f:
    mod_data = json.load(f)
print(len(train_data))
print(len(mod_data))

46
5737


## 1. Review and construct examples

goal - 5-10 examples from train_data, test on mod_data

In [3]:
examples=[]
train_data[0]['abstract']

'Background: Since the appearance of the first case of COVID-19 in Morocco, the cumulative number of reported infectious cases continues to increase and, consequently, the government imposed the containment measure within the country. Our aim is to predict the impact of the compulsory containment on COVID-19 spread. Earlier knowledge of the epidemic characteristics of COVID-19 transmission related to Morocco will be of great interest to establish an optimal plan-of-action to control the epidemic.\n\nMethod: Using a Susceptible-Asymptomatic-Infectious model and the data of reported cumulative confirmed cases in Morocco from March 2nd to April 9, 2020, we determined the basic and control reproduction numbers and we estimated the model parameter values. Furthermore, simulations of different scenarios of containment are performed.\n\nResults: Epidemic characteristics are predicted according to different rates of containment. The basic reproduction number is estimated to be 2.9949, with CI(

In [4]:
text = ("Using a Susceptible-Asymptomatic-Infectious model and the data of reported cumulative confirmed cases "
        "in Morocco from March 2nd to April 9, 2020, we determined the basic and control reproduction numbers "
        "and we estimated the model parameter values.")

head ="susceptible-asymptomatic-infectious model"
head_type="modeling approach"
tail="basic reproduction number"
tail_type="parameter"
relationship="used_to_determine"

examples.append( { "head": head, "head_type":head_type,"relation": relationship,
                   "tail": tail, "tail_type": tail_type})

tail2 = "control reproduction number"
examples.append( { "head": head, "head_type":head_type,"relation": relationship,
                   "tail": tail2, "tail_type": tail_type})


In [5]:
examples

[{'head': 'susceptible-asymptomatic-infectious model',
  'head_type': 'modeling approach',
  'relation': 'used_to_determine',
  'tail': 'basic reproduction number',
  'tail_type': 'parameter'},
 {'head': 'susceptible-asymptomatic-infectious model',
  'head_type': 'modeling approach',
  'relation': 'used_to_determine',
  'tail': 'control reproduction number',
  'tail_type': 'parameter'}]

In [6]:
input2= "The basic reproduction number is estimated to be 2.9949, with CI(2.6729-3.1485)"
output2 =  { 'nodes': [
                {'id':'basic reproduction number','type':'parameter'},
                {'id':'2.9949, with CI(2.6729-3.1485)','type':'value'}            
            ],
             'relationships': [
                { 'source': {'id':'basic reproduction number','type':'parameter'},
                  'destination':  {'id':'2.9949, with CI(2.6729-3.1485)','type':'value'},
                  'type':'parameter_has_value'
                 }
             ]
}


"Different Rates of containment PREDICT epidemic characteristics"
examples.append({'input':input2,'output':output2})

In [7]:
input3 ="Furthermore, a threshold value of containment rate, below which the epidemic duration is postponed, is determined"
output3 ={ 'nodes': [
                {'id':'threshold value of containment','type':'parameter'},
                {'id':'rate below which the epidemic duration is postponed','type':'parameter'}
            ],
             'relationships': [
                { 'source':  {'id':'threshold value of containment','type':'parameter'},
                  'destination': {'id':'rate below which the epidemic duration is postponed','type':'parameter'},
                  'type':'parameter_has_definition'
                 }
             ]
}
examples.append({'input':input3,'output':output3})

In [8]:
input4="Our findings show that the basic reproduction number reflects a high speed of spread of the epidemic"
output4 ={ 'nodes': [
                {'id':'basic reproduction number','type':'parameter'},
                {'id':'high speed of spread of the epidemic'}
            ],
             'relationships': [
                { 'source':     {'id':'basic reproduction number','type':'parameter'},
                  'destination':    {'id':'high speed of spread of the epidemic'},
                  'type':'parameter_has_implications'
                 }
             ]
}
examples.append({'input':input4,'output':output4})

In [9]:
#input6="Furthermore, the compulsory containment can be efficient if more than 73% of population are confined."
#output7="Compulsory containment effiency threshold THRESHOLD_REACHED if more than 73% of population are confined"
#examples.append({'input':input6,'output':output7})

In [10]:
#input7 ="However, even with 90% of containment, the end-time is estimated to happen on July 4th which can be harmful and lead to consequent social-economic damages."
#output8="epidemic end-time EPIDEMIC_END_DATE_ESTIMATE July 4"
#output9="epidemic end_time HAS_IMPLICATIONS harmful and consequent social-economic damages"
#examples.append({'input':input7,'output':output8})
#examples.append({'input':input7,'output':output9})

In [11]:
#input8 = "sensitivity analysis investigation shows that the COVID-19 dynamics depends strongly on the asymptomatic duration as well as the contact and containment rates"
#output10="asymptomatic duration INFLUENCES COVID-19 dynamics"
#output11="contact rates INFLUENCES COVID-19 dynamics"
#output12="containment rates INFLUENCES COVID-19 dynamics"
#examples.append({'input':input8,'output':output10})
#examples.append({'input':input8,'output':output11})
#examples.append({'input':input8,'output':output12})

## try this with examples 

In [12]:
transformer2 = LLMGraphTransformer(llm=llm,user_examples=examples)

self function call is True
after trying sturctured output. self function call is True
val is...True
creating unstructured prompt
.........

.........
.........
[{'head': 'susceptible-asymptomatic-infectious model', 'head_type': 'modeling approach', 'relation': 'used_to_determine', 'tail': 'basic reproduction number', 'tail_type': 'parameter'}, {'head': 'susceptible-asymptomatic-infectious model', 'head_type': 'modeling approach', 'relation': 'used_to_determine', 'tail': 'control reproduction number', 'tail_type': 'parameter'}, {'input': 'The basic reproduction number is estimated to be 2.9949, with CI(2.6729-3.1485)', 'output': {'nodes': [{'id': 'basic reproduction number', 'type': 'parameter'}, {'id': '2.9949, with CI(2.6729-3.1485)', 'type': 'value'}], 'relationships': [{'source': {'id': 'basic reproduction number', 'type': 'parameter'}, 'destination': {'id': '2.9949, with CI(2.6729-3.1485)', 'type': 'value'}, 'type': 'parameter_has_value'}]}}, {'input': 'Furthermore, a threshold val

In [13]:
example_out = [Document(page_content=train_data[0]['abstract'])]
gd = transformer2.convert_to_graph_documents(example_out)
gd

in process_response...
...raw_schema..
{"properties": {"head": {"description": "extracted head entity like Morocco, COVID-19, population.", "title": "Head", "type": "string"}, "head_type": {"description": "type of the extracted head entity like Country, Disease, etc", "title": "Head Type", "type": "string"}, "relation": {"description": "relation between the head and the tail entities", "title": "Relation", "type": "string"}, "tail": {"description": "extracted tail entity like Morocco, COVID-19, population.", "title": "Tail", "type": "string"}, "tail_type": {"description": "type of the extracted tail entity like Country, Disease, etc", "title": "Tail Type", "type": "string"}}, "required": ["head", "head_type", "relation", "tail", "tail_type"]}

[
  {
    "head": "Morocco",
    "head_type": "Country",
    "relation": "affected_by",
    "tail": "COVID-19",
    "tail_type": "Disease"
  },
  {
    "head": "COVID-19",
    "head_type": "Disease",
    "relation": "transmitted_to",
    "tail": 

[GraphDocument(nodes=[Node(id='more than 73% of population are confined', type='Condition', properties={}), Node(id='epidemic duration is postponed', type='Event', properties={}), Node(id='the end-time is estimated to happen on July 4th', type='Consequence', properties={}), Node(id='compulsory containment can be efficient', type='Statement', properties={}), Node(id='sensitivity analysis investigation', type='Study', properties={}), Node(id='the COVID-19 dynamics depends strongly on the asymptomatic duration as well as the contact and containment rates', type='Statement', properties={}), Node(id='COVID-19 dynamics', type='Event', properties={}), Node(id='containment rates', type='Value', properties={}), Node(id='mass testing', type='Action', properties={}), Node(id='rate below which the epidemic duration is postponed', type='Description', properties={}), Node(id='consequent social-economic damages', type='Consequence', properties={}), Node(id='epidemic characteristics of COVID-19 in Mor

In [14]:
gd[0].nodes

[Node(id='more than 73% of population are confined', type='Condition', properties={}),
 Node(id='epidemic duration is postponed', type='Event', properties={}),
 Node(id='the end-time is estimated to happen on July 4th', type='Consequence', properties={}),
 Node(id='compulsory containment can be efficient', type='Statement', properties={}),
 Node(id='sensitivity analysis investigation', type='Study', properties={}),
 Node(id='the COVID-19 dynamics depends strongly on the asymptomatic duration as well as the contact and containment rates', type='Statement', properties={}),
 Node(id='COVID-19 dynamics', type='Event', properties={}),
 Node(id='containment rates', type='Value', properties={}),
 Node(id='mass testing', type='Action', properties={}),
 Node(id='rate below which the epidemic duration is postponed', type='Description', properties={}),
 Node(id='consequent social-economic damages', type='Consequence', properties={}),
 Node(id='epidemic characteristics of COVID-19 in Morocco', typ

In [15]:
gd[0].relationships

[Relationship(source=Node(id='Morocco', type='Country', properties={}), target=Node(id='COVID-19', type='Disease', properties={}), type='affected_by', properties={}),
 Relationship(source=Node(id='COVID-19', type='Disease', properties={}), target=Node(id='population', type='Group', properties={}), type='transmitted_to', properties={}),
 Relationship(source=Node(id='population', type='Group', properties={}), target=Node(id='COVID-19', type='Disease', properties={}), type='affected_by', properties={}),
 Relationship(source=Node(id='Morocco', type='Country', properties={}), target=Node(id='March 2nd to April 9, 2020', type='Time Period', properties={}), type='reported_cases_in', properties={}),
 Relationship(source=Node(id='COVID-19', type='Disease', properties={}), target=Node(id='epidemic', type='Event', properties={}), type='spread_of', properties={}),
 Relationship(source=Node(id='epidemic', type='Event', properties={}), target=Node(id='COVID-19 transmission related to Morocco', type=

## without.

In [16]:
### 4.1 First basic

In [17]:

transformer = LLMGraphTransformer(llm=llm)
graph_documents = transformer.convert_to_graph_documents(example_out)

self function call is True
after trying sturctured output. self function call is True
val is...False

in process_response...


In [18]:
graph_documents

[GraphDocument(nodes=[Node(id='Covid-19', type='Disease', properties={}), Node(id='Morocco', type='Country', properties={}), Node(id='Epidemic', type='Concept', properties={}), Node(id='Containment Measure', type='Measure', properties={}), Node(id='Susceptible-Asymptomatic-Infectious Model', type='Model', properties={})], relationships=[Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Morocco', type='Country', properties={}), type='OCCURRED_IN', properties={}), Relationship(source=Node(id='Morocco', type='Country', properties={}), target=Node(id='Covid-19', type='Disease', properties={}), type='AFFECTED_BY', properties={}), Relationship(source=Node(id='Epidemic', type='Concept', properties={}), target=Node(id='Containment Measure', type='Measure', properties={}), type='RELATED_TO', properties={}), Relationship(source=Node(id='Containment Measure', type='Measure', properties={}), target=Node(id='Susceptible-Asymptomatic-Infectious Model', type='Mod

In [19]:
graph_documents[0].nodes

[Node(id='Covid-19', type='Disease', properties={}),
 Node(id='Morocco', type='Country', properties={}),
 Node(id='Epidemic', type='Concept', properties={}),
 Node(id='Containment Measure', type='Measure', properties={}),
 Node(id='Susceptible-Asymptomatic-Infectious Model', type='Model', properties={})]

In [20]:
graph_documents[0].relationships

[Relationship(source=Node(id='Covid-19', type='Disease', properties={}), target=Node(id='Morocco', type='Country', properties={}), type='OCCURRED_IN', properties={}),
 Relationship(source=Node(id='Morocco', type='Country', properties={}), target=Node(id='Covid-19', type='Disease', properties={}), type='AFFECTED_BY', properties={}),
 Relationship(source=Node(id='Epidemic', type='Concept', properties={}), target=Node(id='Containment Measure', type='Measure', properties={}), type='RELATED_TO', properties={}),
 Relationship(source=Node(id='Containment Measure', type='Measure', properties={}), target=Node(id='Susceptible-Asymptomatic-Infectious Model', type='Model', properties={}), type='USED_FOR', properties={})]